In [1]:
import wandb
wandb.login(key="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")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\thanh\_netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from preprocessing import build_preprocessor, load_and_split
import os
import joblib
import datetime
import numpy as np
from sklearn.model_selection import cross_val_score

SEED = 42

In [3]:
# So sánh các model
X_train, X_test, y_train, y_test = load_and_split('../data/student-mat.csv')
preprocessor = build_preprocessor()

wandb.init(project="student-performance", name="So_sanh_3_mo_hinh")

models = {
    "Linear_Regression": LinearRegression(),
    "Random_Forest": RandomForestRegressor(random_state=SEED),
    "Gradient_Boosting": GradientBoostingRegressor(random_state=SEED)
}
print(">>>>>> So sánh hiệu suất 3 mô hình <<<<<<")
for name, model in models.items():
    pipe = Pipeline([
        ("preprocessing", preprocessor),
        ("model", model)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    r2 = round(r2_score(y_test, y_pred), 4)
    rmse = round(mean_squared_error(y_test, y_pred) ** 0.5, 4)
    mae = round(mean_absolute_error(y_test,y_pred),4)
    
    wandb.log({
        "Model": name,
        "R2": r2,
        "RMSE": rmse,
        "MAE":mae
    })
    print(f"{name:<20} | R2: {r2} | RMSE: {rmse} | MAE: {mae}")

wandb.finish()

>>>>>> So sánh hiệu suất 3 mô hình <<<<<<
Linear_Regression    | R2: 0.7973 | RMSE: 2.0389 | MAE: 1.4377
Random_Forest        | R2: 0.8592 | RMSE: 1.6991 | MAE: 1.079
Gradient_Boosting    | R2: 0.8502 | RMSE: 1.7524 | MAE: 1.1042


MAE,█▁▁
R2,▁█▇
RMSE,█▁▂
MAE,1.1042
Model,Gradient_Boosting
R2,0.8502
RMSE,1.7524


In [4]:
# Thực hiện train sweed
sweep_config = {
    'method': 'grid',
    'name': 'RandomForest-Hyperparameter-Tuning',
    'metric': {
        'name': 'CV_R2',
        'goal': 'maximize'
    },
    'parameters': {
        'n_estimators': {'values': [100, 200, 300]},
        'max_depth': {'values': [5, 10, 20]}
    }
}

def train_sweep():
    with wandb.init() as run:
        config = wandb.config
        
        model = RandomForestRegressor(
            n_estimators=config.n_estimators,
            max_depth=config.max_depth,
            random_state=SEED
        )
        
        pipeline = Pipeline([
            ("preprocessing", preprocessor), #
            ("model", model)
        ])
        
        cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='r2', n_jobs=-1)
        cv_r2 = round(np.mean(cv_scores), 4)
        
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        
        test_r2 = round(r2_score(y_test, y_pred), 4)
        test_rmse = round(mean_squared_error(y_test, y_pred) ** 0.5, 4)
        test_mae = round(mean_absolute_error(y_test, y_pred),4)
        
        wandb.log({
            "CV_R2": cv_r2,
            "Test_R2": test_r2,
            "Test_RMSE": test_rmse,
            "Test_MAE": test_mae
        })
sweep_id = wandb.sweep(sweep_config, project="student-performance")
wandb.agent(sweep_id, function=train_sweep)

Create sweep with ID: rvwohh56
Sweep URL: https://wandb.ai/maiithanhai-open-university-ho-chi-minh-city/student-performance/sweeps/rvwohh56


wandb: Agent Starting Run: qi65qobh with config:
wandb: 	max_depth: 5
wandb: 	n_estimators: 100
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\thanh\_netrc.
wandb: Currently logged in as: maiithanhai (maiithanhai-open-university-ho-chi-minh-city) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


CV_R2,▁
Test_MAE,▁
Test_R2,▁
Test_RMSE,▁
CV_R2,0.8917
Test_MAE,1.0848
Test_R2,0.8611
Test_RMSE,1.6876


wandb: Agent Starting Run: i3jt4wuf with config:
wandb: 	max_depth: 5
wandb: 	n_estimators: 200
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\thanh\_netrc.


CV_R2,▁
Test_MAE,▁
Test_R2,▁
Test_RMSE,▁
CV_R2,0.8895
Test_MAE,1.1061
Test_R2,0.854
Test_RMSE,1.7302


...

wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: jhtkhzys with config:
wandb: 	max_depth: 20
wandb: 	n_estimators: 200
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\thanh\_netrc.


CV_R2,▁
Test_MAE,▁
Test_R2,▁
Test_RMSE,▁
CV_R2,0.8895
Test_MAE,1.1016
Test_R2,0.8522
Test_RMSE,1.741


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: ydz0ryjc with config:
wandb: 	max_depth: 20
wandb: 	n_estimators: 300
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\thanh\_netrc.


CV_R2,▁
Test_MAE,▁
Test_R2,▁
Test_RMSE,▁
CV_R2,0.8904
Test_MAE,1.1024
Test_R2,0.8488
Test_RMSE,1.7606


wandb: Sweep Agent: Waiting for job.
wandb: Sweep Agent: Exiting.


Nhận xét:
- 

In [5]:
BEST_N_ESTIMATORS = 100 
BEST_MAX_DEPTH = 5

final_model = RandomForestRegressor(
    n_estimators=BEST_N_ESTIMATORS, 
    max_depth=BEST_MAX_DEPTH, 
    random_state=SEED
)
final_pipe = Pipeline([("preprocessing", preprocessor), ("model", final_model)])
final_pipe.fit(X_train, y_train)

final_package = {
    "pipeline": final_pipe,
    "metadata": {
        "model_name": "RandomForest_Tuned",
        "best_params": {"n_estimators": BEST_N_ESTIMATORS, "max_depth": BEST_MAX_DEPTH},
        "date_created": datetime.datetime.now().strftime("%Y-%m-%d"),
        "description": "Mô hình dự đoán điểm G3 đã qua tối ưu Sweeps"
    }
}

os.makedirs('../models', exist_ok=True)
joblib.dump(final_package, '../models/best_model_package.joblib')

['../models/best_model_package.joblib']